In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas
from sklearn.preprocessing import StandardScaler

In [ ]:
train_df = pandas.read_csv("/kaggle/input/titanic/train.csv")
test_df = pandas.read_csv("/kaggle/input/titanic/test.csv")
compine = [train_df, test_df]

In [ ]:
train_df.head(5)

In [ ]:
a = ['Pclass', 'Sex', 'Parch', 'SibSp']
p = 0
dic = train_df[[a[p], 'Survived']].groupby(a[p]).mean().sort_values(by = a[p])
dic.plot(kind = 'bar', color = 'b')
plt.show()

In [ ]:
p = 1
dic = train_df[[a[p], 'Survived']].groupby(a[p]).mean().sort_values(by = a[p])
print(dic)
dic.plot(kind = 'bar', color = 'g')
plt.show()

In [ ]:
p = 2
dic = train_df[[a[p], 'Survived']].groupby(a[p]).mean().sort_values(by = a[p])
print(dic)
dic.plot(kind = 'bar', color = 'r')
plt.show()

In [ ]:
p = 3
dic = train_df[[a[p], 'Survived']].groupby(a[p]).mean().sort_values(by = a[p])
print(dic)
dic.plot(kind = 'bar', color = 'y')
plt.show()

In [ ]:
train_df = train_df.drop(['Cabin', 'Ticket', 'PassengerId'], axis = 1)
test_df = test_df.drop(['Cabin', 'Ticket'], axis = 1)
compine = [train_df, test_df]

In [ ]:
train_df.head()

In [ ]:
for dataset in compine:
    titles_list = []
    for name in dataset['Name']:
        title = ''
        for i in range(len(name)):
            if name[i] == ',':
                for j in range(i+1, len(name)):
                    if name[j] == '.':
                        break
                    elif name[j] != ' ':
                        title = title + name[j]
                break
        titles_list.append(title)
    dataset['Title'] = titles_list

In [ ]:
train_df.head()

In [ ]:
train_df = train_df.drop(['Name'], axis = 1)
test_df = test_df.drop(['Name'], axis = 1)
compine = [train_df, test_df]

In [ ]:
train_df.head()

In [ ]:
for dataset in compine:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'theCountess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')

In [ ]:
dic = train_df[['Title', 'Survived']].groupby('Title').mean()
print(dic)
dic.plot(kind = 'bar', color = 'pink')
plt.show()

In [ ]:
train_df.head()

In [ ]:
for dataset in compine:
    dataset['Title'] = dataset['Title'].map({'Mr': 1, 'Rare': 2, 'Master':3, 'Miss':4, 'Mrs':5})
    dataset['Sex'] = dataset['Sex'].map({'male':0, 'female':1})


In [ ]:
train_df.head()

In [ ]:
for dataset in compine:
    probable_ages = np.zeros((2, 3))
    for i in range(2):
        for j in range(3):
            probable_ages[i][j] = dataset[(dataset['Sex'] == i) & (dataset['Pclass'] == j+1)]['Age'].dropna().median()
    for i in range(2):
        for j in range(3):
            dataset.loc[(dataset['Sex'] == i) & (dataset['Pclass'] == j+1) & (dataset['Age'].isnull()), 'Age'] = int(probable_ages[i][j])

In [ ]:
for dataset in compine:
    dataset.loc[(dataset['Age'] <= 16),'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[(dataset['Age'] > 64), 'Age'] = 4

In [ ]:
train_df.head()

In [ ]:
for dataset in compine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
dic = train_df[['FamilySize', 'Survived']].groupby('FamilySize').mean()
print(dic)
dic.plot(kind = 'bar', color = 'teal')
plt.show()

In [ ]:
train_df.head()

In [ ]:
for dataset in compine:
    dataset.loc[(dataset['FamilySize'] == 1), 'IsAlone'] = int(1)
    dataset.loc[(dataset['FamilySize'] > 1), 'IsAlone'] = int(0)

In [ ]:
train_df.head()

In [ ]:
train_df = train_df.drop(['Parch', 'SibSp', 'FamilySize'], axis = 1)
test_df = test_df.drop(['Parch', 'SibSp', 'FamilySize'], axis = 1)
compine = [train_df, test_df]

In [ ]:
train_df.head()

In [ ]:
max_freq = dataset['Embarked'].dropna().mode()[0]
max_freq

In [ ]:
for dataset in compine:
    dataset['Embarked'] = dataset['Embarked'].fillna(max_freq)

In [ ]:
dic = train_df[['Embarked', 'Survived']].groupby('Embarked').mean()
dic.plot(kind = 'bar', color = 'orange')
plt.show()

In [ ]:
train_df = train_df.drop(['Embarked', 'Age'], axis = 1)
test_df = test_df.drop(['Embarked', 'Age'], axis = 1)
compine = [train_df, test_df]

In [ ]:
test_df['Fare'].fillna(test_df['Fare'].dropna().median(), inplace=True)
test_df.head()

In [ ]:
for dataset in compine:
    mx = dataset['Fare'].max()
    dataset['Fare'] = (dataset['Fare']/mx)

In [ ]:
train_df.info()

In [ ]:
test_df.head()

In [ ]:
X_train = train_df.drop('Survived', axis = 1)
y_train = train_df['Survived']
X_test = test_df.drop(['PassengerId'], axis = 1).copy()
X_test = StandardScaler().fit_transform(X_test)

In [ ]:
Acc = {}
X_train.shape, y_train.shape, X_test.shape

In [ ]:
from sklearn.linear_model import SGDClassifier
sgd = SGDClassifier()
sgd.fit(X_train, y_train)
ac_sgd = round(sgd.score(X_train, y_train)*100,2)
Acc['SGDC'] = ac_sgd
print('Stocastic Gradient Descent Accuracy: ', ac_sgd)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, y_train)
acc_knn = knn.score(X_train, y_train)
acc_knn = round(acc_knn*100, 2)
Acc['KNN'] = acc_knn
print('Kth Nearest Neighbors Accuracy: ', acc_knn)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decision_tree = DecisionTreeClassifier()
decision_tree.fit(X_train, y_train)
acc_decision_tree = decision_tree.score(X_train, y_train)
acc_decision_tree = round(acc_decision_tree*100, 2)
Acc['DecisionTree'] = acc_decision_tree
print('Decision Tree Accuracy: ', acc_decision_tree)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier(n_estimators = 20)
RF.fit(X_train, y_train)
acc_RF = RF.score(X_train, y_train)
acc_RF = round(acc_RF*100, 2)
Acc['RandomForest'] = acc_RF
print('Random Forest Accuracy: ', acc_RF)

In [ ]:
from sklearn.naive_bayes import GaussianNB
NB = GaussianNB()
NB.fit(X_train, y_train)
acc_NB = NB.score(X_train, y_train)
acc_NB = round(acc_NB *100, 2)
Acc['NaiveBayes'] = acc_NB
print("NaiveBayes Accuracy: ", acc_NB)

In [ ]:
from sklearn.svm import SVC
SV_clf = SVC(kernel = 'poly', degree = 5)
SV_clf.fit(X_train, y_train)
acc_SV = SV_clf.score(X_train, y_train)
acc_SV = round(acc_SV *100, 2)
Acc['NaiveBayes'] = acc_SV
print("NaiveBayes Accuracy: ", acc_SV)

In [ ]:
mp = pandas.DataFrame(Acc.items())
mp = pandas.DataFrame({'Model':mp[0], 'Accuracy':mp[1]})
mp.plot(x = 'Model', y = 'Accuracy', kind = 'bar', color = 'purple')
mp

In [ ]:
import seaborn as sns
sns.heatmap(train_df.corr(), annot = True)
plt.show()

In [ ]:
y_pred = SV_clf.predict(X_test)
y_pred[:5]

In [ ]:
submission = pandas.DataFrame({
        "PassengerId": test_df["PassengerId"],
        "Survived": y_pred
    })

In [ ]:
submission.to_csv('submission.csv', index = False)

In [ ]:
submission